---
title: Tracks widget
---

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from geneinfo.widget import Tracks
from geneinfo.coords import chromosome_lengths, gene_coords, gene_labels
from geneinfo.genelist import GeneList as glist
from geneinfo.genelist import GeneListCollection
from vscodenb import set_vscode_theme
import seaborn as sns
sns.set_palette('tab10')
set_vscode_theme()

Some example data:

In [ ]:
chrom_sizes = dict(chromosome_lengths(assembly='hg38'))
rng = np.random.default_rng(42)
quant_rows = []
n_points = 10000
for chrom, csz in chrom_sizes.items():
    positions = np.sort(rng.integers(0, csz, n_points))
    fst = np.abs(rng.normal(0.05, 0.08, n_points)).clip(0, 1)
    rate = np.cumsum(rng.normal(0, 0.3, n_points))
    rate = (rate - rate.min()) / (rate.max() - rate.min()) * 5 + 0.5
    mid = np.cumsum(rng.normal(0, 0.15, n_points))
    eig = np.sin(positions / 1e7 * 2 * np.pi) + rng.normal(0, 0.2, n_points)
    ci_lo = mid - rng.exponential(0.3, n_points)
    ci_hi = mid + rng.exponential(0.3, n_points) + 10
    # depth = rng.poisson(30, n_points).astype(float)
    depth = ci_hi

    for pop in ['AFR', 'EUR']:
        noise = rng.normal(0, 0.02, n_points)
        for i in range(n_points):
            quant_rows.append({
                'chrom': chrom, 'pos': int(positions[i]),
                'fst': float(fst[i] + noise[i]),
                'rate': float(rate[i] + noise[i] * 10) + (5 if pop == 'EUR' else 0),
                'eig': float(eig[i] + noise[i]),
                'ci_lo': float(ci_lo[i]) + (15 if pop == 'EUR' else 0), 
                'ci_hi': float(ci_hi[i]) + (15 if pop == 'EUR' else 0), 
                'depth': float(depth[i] + rng.normal(0, 3)),
                'pop': pop,
            })
dummy_data = pd.DataFrame(quant_rows)  

rng = np.random.default_rng(7)
arc_rows = []
for chrom, csz in list(chrom_sizes.items()):
    n = 30
    p1 = rng.integers(0, csz, n)
    p2 = p1 + rng.integers(csz // 100, csz // 5, n)
    p2 = np.clip(p2, 0, csz - 1)
    kind = rng.choice(['cis', 'trans'], size=n, p=[0.7, 0.3])
    for a, b, k in zip(p1, p2, kind):
        arc_rows.append({'chrom': chrom, 'pos1': int(a), 'pos2': int(b), 'kind': k})
dummy_arc_data = pd.DataFrame(arc_rows)

sheet = GeneListCollection(google_sheet='1JSjSLuto3jqdEnnG7JqzeC_1pUZw76n7XueVAYrUOpk')
hybrid_diff_exp_genes = sheet.get('hybridDEG')

def gene_view(name, span=1000000):
    chrom, start, end, _ = gene_coords(name, assembly='hg38')[0]
    return dict(chrom=chrom, center=(start+end)/2, window=span)

def gene_positions(names):
    positions = defaultdict(list)
    for chrom, pos, _ in gene_labels(names, assembly='hg38'):
        positions[chrom].append(pos)
    return positions

Archaic segments:

In [ ]:
segments = pd.read_parquet('segments.parquet')
print(f'{len(segments):,} segments from {segments["ind"].nunique()} individuals')

Build tracks (takes a while to build with small windows). See below for controls.

In [ ]:
tracks = (
    Tracks('hg38')
    .add_segment_track(segments, 'Ancestry', group_by='ancestry', individual_col='ind', height=80, tip_fmt=False, stack=True, density_windows=(512, 2048, 8192))  
    .add_heatmap_track(segments, "Ancestry", individual_col="ind", group_by="reg", value_col="ancestry", windows="10kb").legend("Ancestry")
   .add_gene_track(highlight=hybrid_diff_exp_genes)
)
tracks

**Toolbar (left-to-right):**
- Chromosome dropdown: switch chromosome.
- Position box: type a coordinate (`50000000`) or range (`chr1:1e6-2e6`) and press Enter. Invalid input flashes the box red with a tooltip hint.
- **＋ / −** zooms in / out (cursor-anchored in the plot).
- **«  ‹  ›  »** pans left / right by 90 % / 50 % of the view.
- **⌂** resets to the full chromosome.
- **⟲** recomputes heatmap(s) at the current view for higher resolution. Shows a spinner while rebin is in flight. *(heatmap tracks only)*
- **◱** restores the whole-chromosome heatmap view. *(heatmap tracks only)*
- **A** shows all gene labels when fewer than 100 genes are visible. *(gene tracks only)*
- **⧉** snapshots the current view. Flashes **✓** if copied to the clipboard, **⇓** if downloaded as a PNG fallback, **!** on failure.
- **?** helps popover listing mouse, keyboard, and toolbar controls. `Esc` or click-outside to dismiss.
- **▬ segments / ▒ density** badge — LOD indicator.

**Mouse:**
- Drag on the plot to pan.
- Scroll wheel** to zoom (anchored on the cursor's genomic position).
- Double-click to zoom in 2× centred on the click.
- Hover for a context-sensitive tooltip (gene name, y-value, group, individual, ...).

**Keyboard (when the widget has focus):**
- **← / →** — pan by 15 % of the current view.
- **+ / =** — zoom in 1.5×.  **−** — zoom out 1.5×.

Documentation: https://munch-group.org/geneinfo/pages/tracks_widget.html

In [ ]:
segments_subset = segments.loc[(segments.chrom == 'chrX') & (segments.reg == 'EUR')]#.groupby(['reg', 'chrom']).apply(lambda df: df.head(10000)).reset_index()

ECH_regions = {'chrX': [
  (19781882, 19881882),  (21181882, 21281882),
  (36181883, 36381885), (37440747, 37840747),
  (50057375, 50057403), (51057154, 51657066),
  (54173567, 54373567), (65480120, 65680139),
  (73780165, 74080165), (74680165, 75080165),
  (77644510, 77944503), (99445002, 99545002),
  (111456772, 111856772), (114765448, 115065437),
  (127866021, 127966021), (130566026, 130966026),
  (132165972, 132465972), (133465972, 133765973),
  (154771725, 155171725)]}

gene_pos = gene_positions(hybrid_diff_exp_genes)

In [ ]:
tracks = (
    Tracks('hg38')

    # your own data
    .clear_spans().add_spans(ECH_regions, color='magenta')
    .clear_vlines().add_vlines(gene_pos)
    .add_point_track(dummy_data, 'Scatter', x='pos', y='fst', group_by='pop', tip_fmt=False)
    .add_line_track(dummy_data, 'Lines', x='pos', y='rate', group_by='pop', step='post', tip_fmt=False)
    .add_fill_track(dummy_data, 'Fill btwn', x='pos', y_hi='ci_hi', y_lo='ci_lo', group_by='pop', tip_fmt=False)
    .add_arc_track(dummy_arc_data, 'Interactions', group_by='kind', line_width=1, alpha=0.85)
    .add_fill_track(dummy_data, 'Fill baseln', x='pos', y='eig', group_by='pop', step='post', tip_fmt=False)
    .add_histogram_track(dummy_data, 'Histogram', x='pos', y='depth', group_by='pop', stack=True, tip_fmt=False)

    # USCC tracks
    .add_ucsc_track('recombAvg', kind='line', color='C0', chrom='chrX')
    .add_ucsc_track('gnomadConstraint', kind='point', color='C4', chrom='chrX')
    .add_gene_track(highlight=hybrid_diff_exp_genes)

    # segment subset
    .add_segment_track(segments_subset, 'Ancestry', group_by='ancestry', individual_col='ind', tip_fmt=False, stack=True, height=50, density_windows=(512, 2048, 8192))  
    .add_heatmap_track(segments_subset, "Ancestry", individual_col="ind", group_by="reg", value_col="ancestry", windows="10kb").legend("Ancestry")
)
tracks.zoom_to(chrom='chrX')